In [1]:
# Step 1: Load Dataset
from datasets import load_dataset
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
# Load AG News dataset from Wangrongsheng
dataset = load_dataset("Wangrongsheng/ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [3]:
# Extract text and labels
train_texts = [item["text"] for item in train_data]
train_labels = [item["label"] for item in train_data]
test_texts = [item["text"] for item in test_data]
test_labels = [item["label"] for item in test_data]

In [4]:
# Step 2: Clean and Normalize Text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)  # keep only letters and spaces
    return text

train_texts = [clean_text(t) for t in train_texts]
test_texts = [clean_text(t) for t in test_texts]

In [5]:
# Step 3: Tokenize
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

In [6]:
# Step 4: Pad Sequences
max_length = 50
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding="post", truncating="post")
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding="post", truncating="post")

In [7]:
# Step 5: One-hot Encode Labels
num_classes = len(set(train_labels))
train_labels_cat = to_categorical(train_labels, num_classes=num_classes)
test_labels_cat = to_categorical(test_labels, num_classes=num_classes)

In [8]:
# Step 6: Define RNN Model
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=max_length),
    SimpleRNN(64),
    Dense(num_classes, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# Step 7: Compile and Train
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_padded, train_labels_cat, epochs=5, batch_size=128, validation_split=0.2)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.8121 - loss: 0.5426 - val_accuracy: 0.8633 - val_loss: 0.4308
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8937 - loss: 0.3395 - val_accuracy: 0.8577 - val_loss: 0.4370
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9139 - loss: 0.2747 - val_accuracy: 0.8730 - val_loss: 0.3825
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9232 - loss: 0.2405 - val_accuracy: 0.8760 - val_loss: 0.3841
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9322 - loss: 0.2122 - val_accuracy: 0.8757 - val_loss: 0.3817


In [10]:
# Step 8: Evaluate
loss, accuracy = model.evaluate(test_padded, test_labels_cat)
print(f"Test Accuracy: {accuracy:.4f}")

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8862 - loss: 0.3551
Test Accuracy: 0.8862
